In [152]:
from utils.random_forest_utils.rf_preprocessing_utils import WindowAlgPreprocessor

rf_preprocessor_clamping = WindowAlgPreprocessor(sensors_path="../../../../data/ml/label_df/clamping_features.csv", target_path="../../../../data/ml/targets.csv")
sensors_df_clamping, target_df_clamping = rf_preprocessor_clamping.read_data()
sensors_df_clamping = rf_preprocessor_clamping.feature_selection()
rf_preprocessor_clamping.normalize_angle()
rf_preprocessor_bending = WindowAlgPreprocessor(sensors_path="../../../../data/ml/label_df/bending_features.csv", target_path="../../../../data/ml/targets.csv")
sensors_df_bending, target_df_bending = rf_preprocessor_bending.read_data()
sensors_df_bending = rf_preprocessor_bending.feature_selection()
rf_preprocessor_bending.normalize_angle()
rf_preprocessor_declamping = WindowAlgPreprocessor(sensors_path="../../../../data/ml/label_df/declamping_features.csv", target_path="../../../../data/ml/targets.csv")
sensors_df_declamping, target_df_declamping = rf_preprocessor_declamping.read_data()
sensors_df_declamping = rf_preprocessor_declamping.feature_selection()
rf_preprocessor_declamping.normalize_angle()

,Experiment_ID,Angle[degree]ORDistance[mm],Secondary-axis [mm],Main-axis [mm],Out-of-roundness [-],Collapse [mm]
0,2,0.000000,0.900079,0.313091,0.999954,0.313091
1,2,0.022017,0.908296,0.425584,0.834481,0.425584
2,2,0.044033,0.905549,0.591178,0.582994,0.591178
3,2,0.066050,0.901145,0.751356,0.338801,0.751356
4,2,0.088067,0.909703,0.867774,0.167582,0.867774
...,...,...,...,...,...,...
14558,318,0.902686,0.928583,0.029142,0.997804,0.029142
14559,318,0.924703,0.922565,0.025609,1.000000,0.025609
14560,318,0.946720,0.921322,0.032466,0.992607,0.032466
14561,318,0.968736,0.925369,0.051137,0.974276,0.051137


In [153]:
def normalize_experiment(group, n=46):
    if len(group) > n:
        # Just take the first 46 rows
        return group.iloc[:n].copy()
    else:
        # Already 46 rows
        return group.copy()

# Apply to each experiment
df_normalized_clamping = target_df_clamping.groupby('Experiment_ID', group_keys=False).apply(normalize_experiment, n=46)
df_normalized_clamping = df_normalized_clamping.reset_index(drop=True)
df_normalized_bending = target_df_bending.groupby('Experiment_ID', group_keys=False).apply(normalize_experiment, n=46)
df_normalized_bending = df_normalized_bending.reset_index(drop=True)
df_normalized_declamping = target_df_declamping.groupby('Experiment_ID', group_keys=False).apply(normalize_experiment, n=46)
df_normalized_declamping = df_normalized_declamping.reset_index(drop=True)

In [154]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis, entropy

def resample_experiment_fast(group, n=46, metric='mean'):
    """
    Optimized resampling function using vectorized operations.
    Up to 10-100x faster than the original implementation.
    """
    # Sort by time
    group = group.sort_values('Time_[s]')
    time_col = group['Time_[s]'].values
    
    # Assign each row to a time bin
    time_bins = np.linspace(time_col.min(), time_col.max(), n + 1)
    bin_indices = np.digitize(time_col, time_bins[:-1]) - 1
    bin_indices = np.clip(bin_indices, 0, n - 1)
    
    # Get experiment ID
    exp_id = group['Experiment_ID'].iloc[0]
    
    # Select numeric columns only
    cols_to_process = [col for col in group.columns 
                       if col not in ['Time_[s]', 'Experiment_ID']]
    
    results = []
    
    # Process each bin
    for bin_idx in range(n):
        mask = bin_indices == bin_idx
        if not mask.any():
            continue
            
        row_data = {'Experiment_ID': exp_id}
        
        for col in cols_to_process:
            values = group[col].values[mask]
            if len(values) == 0:
                continue
            
            # Compute metric using vectorized operations
            if metric == 'mean':
                row_data[f'{col}_mean'] = values.mean()
            elif metric == 'median':
                row_data[f'{col}_median'] = np.median(values)
            elif metric == 'min':
                row_data[f'{col}_min'] = values.min()
            elif metric == 'max':
                row_data[f'{col}_max'] = values.max()
            elif metric == 'range':
                row_data[f'{col}_range'] = values.ptp()
            elif metric == 'std':
                row_data[f'{col}_std'] = values.std()
            elif metric == 'var':
                row_data[f'{col}_var'] = values.var()
            elif metric == 'mad':
                row_data[f'{col}_mad'] = np.abs(values - values.mean()).mean()
            elif metric == 'rms':
                row_data[f'{col}_rms'] = np.sqrt((values ** 2).mean())
            elif metric == 'skew':
                row_data[f'{col}_skew'] = skew(values)
            elif metric == 'kurtosis':
                row_data[f'{col}_kurtosis'] = kurtosis(values)
            elif metric == 'energy':
                row_data[f'{col}_energy'] = (values ** 2).sum()
            elif metric == 'entropy':
                abs_vals = np.abs(values)
                probs = abs_vals / (abs_vals.sum() + 1e-12)
                row_data[f'{col}_entropy'] = entropy(probs + 1e-12)
            elif metric == 'cv':
                row_data[f'{col}_cv'] = values.std() / (values.mean() + 1e-12)
            elif metric == 'iqr':
                row_data[f'{col}_iqr'] = np.percentile(values, 75) - np.percentile(values, 25)
            elif metric == 'p25':
                row_data[f'{col}_p25'] = np.percentile(values, 25)
            elif metric == 'p75':
                row_data[f'{col}_p75'] = np.percentile(values, 75)
            elif metric == 'trend_slope':
                if len(values) > 1:
                    row_data[f'{col}_trend_slope'] = np.polyfit(np.arange(len(values)), values, 1)[0]
                else:
                    row_data[f'{col}_trend_slope'] = 0
        
        results.append(row_data)
    
    return pd.DataFrame(results)


# Alternative: Ultra-fast version using pandas groupby (even faster for 'mean', 'std', 'min', 'max')
def resample_experiment_ultrafast(group, n=46, metric='mean'):
    """
    Ultra-optimized version using pandas groupby operations.
    Works best for basic metrics like mean, std, min, max, median.
    """
    group = group.sort_values('Time_[s]')
    time_col = group['Time_[s]'].values
    
    # Assign bins
    time_bins = np.linspace(time_col.min(), time_col.max(), n + 1)
    group['_bin'] = np.digitize(time_col, time_bins[:-1]) - 1
    group['_bin'] = group['_bin'].clip(0, n - 1)
    
    # Select columns to aggregate
    cols_to_agg = [col for col in group.columns 
                   if col not in ['Time_[s]', 'Experiment_ID', '_bin']]
    
    # Map metric to pandas aggregation function
    agg_func_map = {
        'mean': 'mean',
        'median': 'median',
        'min': 'min',
        'max': 'max',
        'std': 'std',
        'var': 'var',
        'sum': 'sum'
    }
    
    if metric in agg_func_map:
        # Use fast pandas groupby
        result = group.groupby('_bin')[cols_to_agg].agg(agg_func_map[metric])
        result = result.add_suffix(f'_{metric}')
        result['Experiment_ID'] = group['Experiment_ID'].iloc[0]
        return result.reset_index(drop=True)
    else:
        # Fall back to custom implementation
        return resample_experiment_fast(group.drop('_bin', axis=1), n, metric)



In [155]:
df_bending = sensors_df_bending.reset_index()
df_resampled_bending = (
    df_bending.groupby('Experiment_ID', group_keys=False)
    .apply(lambda g: resample_experiment_ultrafast(g, n=40, metric='mean'))
    .reset_index(drop=True)
)


# Apply to all datasets
df_clamping = sensors_df_clamping.reset_index()
df_resampled_clamping = (
    df_clamping.groupby('Experiment_ID', group_keys=False)
    .apply(lambda g: resample_experiment_ultrafast(g, n=200, metric='mean'))
    .reset_index(drop=True)
)


df_declamping = sensors_df_declamping.reset_index()
df_resampled_declamping = (
    df_declamping.groupby('Experiment_ID', group_keys=False)
    .apply(lambda g: resample_experiment_ultrafast(g, n=200, metric='mean'))
    .reset_index(drop=True)
)


In [156]:
from utils.lstm_utils.lstm_preprocessing_utils import WindowAlgPreprocessor

lstm_preprocessor = WindowAlgPreprocessor(sensors_path="../../../../data/ml/features_machine_and_movement_complete.csv", target_path="../../../../data/ml/targets_movement_complete.csv")
sensors_df, target_df = lstm_preprocessor.read_data()

In [157]:
sensors_df = sensors_df.reset_index()
sensor_data_100 = sensors_df[(sensors_df["Experiment_ID"]==100)]
time_stamp_clamping_start = sensors_df_clamping[sensors_df_clamping["Experiment_ID"]==100].index[0]
time_stamp_bending_start = sensors_df_bending[sensors_df_bending["Experiment_ID"]==100].index[0]
time_stamp_declamping_start = sensors_df_declamping[sensors_df_declamping["Experiment_ID"]==100].index[0]
time_stamp_declamping_end = sensors_df_declamping[sensors_df_declamping["Experiment_ID"]==100].index[-1]

# Find the local index (row number) in `a`
idx_clamping_start = sensor_data_100.index.get_loc(sensor_data_100[sensor_data_100["Time_[s]"] == time_stamp_clamping_start].index[0])
idx_bending_start = sensor_data_100.index.get_loc(sensor_data_100[sensor_data_100["Time_[s]"] == time_stamp_bending_start].index[0])
idx_declamping_start = sensor_data_100.index.get_loc(sensor_data_100[sensor_data_100["Time_[s]"] == time_stamp_declamping_start].index[0])
idx_declamping_end = sensor_data_100.index.get_loc(sensor_data_100[sensor_data_100["Time_[s]"] == time_stamp_declamping_end].index[0])

annot_timesteps = [idx_clamping_start-20, idx_bending_start-50, idx_declamping_start, idx_declamping_end-50]


In [158]:
N = len(sensor_data_100)
n_samples_total = 200
annot_timesteps = [int(idx / N * n_samples_total) for idx in annot_timesteps]


In [159]:
sensors_df_aoi = sensors_df[(sensors_df.index > 27) & (sensors_df.index < 65)]

df_resampled = (
    sensors_df.groupby('Experiment_ID', group_keys=False)
    .apply(lambda g: resample_experiment_ultrafast(g, n=n_samples_total, metric='mean'))
    .reset_index(drop=True)
)
sensors_df_aoi = sensors_df_aoi.reset_index()
sensors_df_aoi_resampled = (
    sensors_df_aoi.groupby('Experiment_ID', group_keys=False)
    .apply(lambda g: resample_experiment_ultrafast(g, n=600, metric='mean'))
    .reset_index(drop=True)
)

In [160]:
angle_idx_start, angle_idx_end, angle_step = 0, 46, 1
feature_idx_start, feature_idx_end = 1, 5
machine_part = "DECLAMPING"

In [161]:
columns = list(target_df_bending.columns[1:])
feature_columns = columns[feature_idx_start: feature_idx_end]

In [162]:
X_clamping = rf_preprocessor_clamping.group_and_pad(df_resampled_clamping, group_col="Experiment_ID")
Y_clamping = rf_preprocessor_clamping.group_and_pad(df_normalized_clamping, group_col="Experiment_ID")[:,angle_idx_start:angle_idx_end: angle_step, feature_idx_start: feature_idx_end]

X_bending = rf_preprocessor_bending.group_and_pad(df_resampled_bending, group_col="Experiment_ID")
Y_bending = rf_preprocessor_bending.group_and_pad(df_normalized_bending, group_col="Experiment_ID")[:,angle_idx_start:angle_idx_end: angle_step, feature_idx_start: feature_idx_end]

X_declamping = rf_preprocessor_declamping.group_and_pad(df_resampled_declamping, group_col="Experiment_ID")
Y_declamping = rf_preprocessor_declamping.group_and_pad(df_normalized_declamping, group_col="Experiment_ID")[55: ,angle_idx_start:angle_idx_end: angle_step, feature_idx_start: feature_idx_end]

X_total = rf_preprocessor_declamping.group_and_pad(df_resampled, group_col="Experiment_ID")

X_aoi = rf_preprocessor_declamping.group_and_pad(sensors_df_aoi_resampled, group_col="Experiment_ID")

In [163]:
import torch

if machine_part == "BENDING":
    X_seq = torch.from_numpy(X_bending).float()
    Y_target = torch.from_numpy(Y_bending).float()
    feature_names = list(df_resampled_bending.columns[:-1])
elif machine_part == "DECLAMPING":
    X_seq = torch.from_numpy(X_declamping).float()
    Y_target = torch.from_numpy(Y_declamping).float()
    feature_names = list(df_resampled_declamping.columns[:-1])
elif machine_part == "CLAMPING":
    X_seq = torch.from_numpy(X_clamping).float()
    Y_target = torch.from_numpy(Y_clamping).float()
    feature_names = list(df_resampled_clamping.columns[:-1])
elif machine_part == "ALL":
    X_seq = torch.from_numpy(X_total).float()
    Y_target = torch.from_numpy(Y_clamping).float()
    feature_names = list(df_resampled.columns[:-1])
elif machine_part == "AOI":
    X_seq = torch.from_numpy(X_aoi).float()
    Y_target = torch.from_numpy(Y_clamping).float()
else:
    raise Exception("Shoud be between CLAMPING, BENDING, DECLAMPING")

In [164]:
import matplotlib.pyplot as plt
from matplotlib import rcParams

def plot_selected_features_with_attn_heatmap(sensor_data, feature_names, attn_mean, attn_path, sample_idx=100, figsize=(25, 12)):
    """
    Plots selected features with attention heatmap at the bottom.
    Includes legend for the top plot on the right side.
    Enhanced with beautiful styling and improved aesthetics.
    """
    # Set beautiful style parameters
    rcParams['font.family'] = 'sans-serif'
    rcParams['font.size'] = 10
    
    # Remove '_mean' from all feature names
    cleaned_feature_names = [name.replace('_mean', '') for name in feature_names]
    
    # Create figure with subplots
    fig = plt.figure(figsize=figsize, facecolor='white')
    fig.clf()
    gs = fig.add_gridspec(2, 1, height_ratios=[2, 1], hspace=0.25, wspace=0.3)
    ax_main = fig.add_subplot(gs[0])
    ax_heatmap = fig.add_subplot(gs[1])
    
    # Get the data for the selected sample
    sample_data = sensor_data[sample_idx, :, :]
    main_timesteps = sample_data.shape[0]
    n_attention_heads = attn_mean.shape[0]
    attn_timesteps = attn_mean.shape[1]
    
    print(f"Main plot timesteps: {main_timesteps}")
    print(f"Attention original shape: {attn_mean.shape}")
    
    # --- MAIN PLOT WITH LEGEND ---
    # Use a sophisticated color palette
    colors = plt.cm.tab20(np.linspace(0, 1, len(cleaned_feature_names)))
    
    # Plot each feature with enhanced styling
    for i, (feature_name, color) in enumerate(zip(cleaned_feature_names, colors)):
        ax_main.plot(sample_data[:, i], 
                    color=color, 
                    linewidth=2.5,
                    alpha=0.85,
                    label=feature_name,
                    marker='o',
                    markersize=3,
                    markevery=max(1, main_timesteps//20))  # Smart marker placement
    
    # Style main plot
    ax_main.set_xlabel("Time Step", fontsize=12, fontweight='bold', labelpad=10)
    ax_main.set_ylabel("Feature Value", fontsize=12, fontweight='bold', labelpad=10)
    ax_main.grid(True, alpha=0.2, linestyle='--', linewidth=0.8, color='gray')
    ax_main.set_axisbelow(True)
    
    # Remove top and right spines for cleaner look
    ax_main.spines['top'].set_visible(False)
    ax_main.spines['right'].set_visible(False)
    ax_main.spines['left'].set_linewidth(1.2)
    ax_main.spines['bottom'].set_linewidth(1.2)
    ax_main.spines['left'].set_color('#333333')
    ax_main.spines['bottom'].set_color('#333333')
    
    if machine_part == "ALL":      
        annot_labels = ["Start-Clamping", "Start-Bending", "Start-Declamping", "End-Clamping"]   # Optional short labels

        for ts, label in zip(annot_timesteps, annot_labels):
            # Vertical line for visibility
            ax_main.axvline(ts, color='black', linestyle='--', linewidth=1.2, alpha=0.7)
            
            # Annotated text placed slightly above the data region
            ax_main.annotate(
                label,
                xy=(ts, sample_data[:, :].max()),    # anchor at top of plot
                xytext=(0, 10),                      # offset upward
                textcoords='offset points',
                ha='center',
                va='bottom',
                fontsize=11,
                fontweight='bold',
                color='black',
                bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='gray', lw=0.8)
            )
    
    if machine_part == "AOI":      
        annot_labels = ["Start-Bending", "Start-Declamping"]   # Optional short labels

        for ts, label in zip(annot_timesteps[1:3], annot_labels):
            # Vertical line for visibility
            ax_main.axvline(ts, color='black', linestyle='--', linewidth=1.2, alpha=0.7)
            
            # Annotated text placed slightly above the data region
            ax_main.annotate(
                label,
                xy=(ts, sample_data[:, :].max()),    # anchor at top of plot
                xytext=(0, 10),                      # offset upward
                textcoords='offset points',
                ha='center',
                va='bottom',
                fontsize=11,
                fontweight='bold',
                color='black',
                bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='gray', lw=0.8)
            )
    
    ax_main.set_xlim(0, main_timesteps-1)
    ax_main.set_facecolor('#f9f9f9')
    ax_main.set_title("Sensor Data Over Time", fontsize=14, fontweight='bold', pad=15)
    
    # Add legend with enhanced styling
    legend = ax_main.legend(bbox_to_anchor=(1.02, 1), 
                           loc='upper left', 
                           borderaxespad=0.,
                           frameon=True,
                           fancybox=True,
                           shadow=True,
                           fontsize=10,
                           framealpha=0.95,
                           edgecolor='#cccccc')
    legend.get_frame().set_facecolor('white')
    legend.get_frame().set_linewidth(1.2)
    
    # --- ATTENTION HEATMAP ---
    # Ensure heatmap has exactly the same number of timesteps as main plot
    if attn_timesteps != main_timesteps:
        print(f"Resizing attention from {attn_timesteps} to {main_timesteps} timesteps")
        attn_data_resized = np.zeros((n_attention_heads, main_timesteps))
        for i in range(n_attention_heads):
            x_original = np.arange(attn_timesteps)
            x_target = np.linspace(0, attn_timesteps-1, main_timesteps)
            attn_data_resized[i] = np.interp(x_target, x_original, attn_mean[i])
        attn_data = attn_data_resized
    else:
        attn_data = attn_mean
    
    print(f"Attention final shape: {attn_data.shape}")
    
    # Create enhanced heatmap
    im = ax_heatmap.imshow(attn_data, 
                          aspect='auto', 
                          cmap='magma',  # More visually appealing colormap
                          interpolation='bilinear',
                          extent=[0, main_timesteps-1, 0, n_attention_heads-1])
    
    # Style heatmap
    ax_heatmap.set_xlabel("Time Step", fontsize=12, fontweight='bold', labelpad=10)
    ax_heatmap.set_ylabel("Attention Head", fontsize=9, fontweight='bold', labelpad=10)
    
    ax_heatmap.set_yticks(np.arange(n_attention_heads))
    # Reverse the label order
    ax_heatmap.set_yticklabels([f'{i+1}' for i in reversed(range(n_attention_heads))], fontsize=5)

    ax_heatmap.set_xlim(0, main_timesteps-1)
    ax_heatmap.set_facecolor('white')
    ax_heatmap.set_title("Attention Head Intensity", fontsize=14, fontweight='bold', pad=15)
    
    # Remove spines for cleaner look
    ax_heatmap.spines['top'].set_visible(False)
    ax_heatmap.spines['right'].set_visible(False)
    ax_heatmap.spines['left'].set_linewidth(1.2)
    ax_heatmap.spines['bottom'].set_linewidth(1.2)
    ax_heatmap.spines['left'].set_color('#333333')
    ax_heatmap.spines['bottom'].set_color('#333333')
    
    # Add colorbar with enhanced styling
    cbar = plt.colorbar(im, ax=ax_heatmap, shrink=0.9, pad=0.02)
    cbar.set_label('Attention Weight', fontsize=11, fontweight='bold', labelpad=10)
    cbar.ax.tick_params(labelsize=9)
    cbar.outline.set_linewidth(1.2)
    
    # Fine-tune layout
    plt.tight_layout()
    
    # Get positions for alignment
    pos_main = ax_main.get_position()
    pos_heat = ax_heatmap.get_position()
    
    # Make heatmap width match main plot width
    ax_heatmap.set_position([pos_heat.x0, pos_heat.y0, pos_main.width, pos_heat.height])
    
    # Reposition colorbar to align properly
    cbar.ax.set_position([pos_main.x0 + pos_main.width + 0.02, 
                         pos_heat.y0, 
                         0.015, 
                         pos_heat.height])
    
    fig.savefig(attn_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    

In [165]:
N_EXPERIMENTS, TIMESTEPS_IN, FEATURES_IN = X_seq.shape
_, PREDICTIONS_OUT, FEATURES_OUT = Y_target.shape


# Apply selection to X_seq
FEATURES_IN = X_seq.shape[2]

In [166]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import random
import warnings
import time
import mlflow
import mlflow.pytorch
from pathlib import Path
from datetime import datetime
import mlflow

if mlflow.active_run() is not None:
    mlflow.end_run()

# Seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# -------------------------------
# MLFLOW SETUP
# -------------------------------
mlflow.set_experiment(f"LSTM_Attention_{machine_part}")
mlflow.set_tracking_uri("mlruns")  # Local tracking

EXPERIMENT_DESCRIPTION = f"""

************************************
Complete Data
************************************
{machine_part} PART - LSTM Attention Model
==========================================
INPUT:
N_EXPERIMENTS, N_TIMESTEPS_IN, N_FEATURES_IN = ({N_EXPERIMENTS}, {TIMESTEPS_IN}, {FEATURES_IN})
OUTPUT:
ANGLE_INDEX: {angle_idx_start}-{angle_idx_end}  -> step: {angle_step}
GEOMETRY_FEATURES: {feature_columns}
"""

warnings.filterwarnings("ignore")

# -------------------------------
# Dataset & Dynamic Model
# -------------------------------
class ProcessDataset(Dataset):
    def __init__(self, X, Y):
        self.X = X.float()
        self.Y = Y.float()
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.Y[idx]

class SimpleMLPAttention(nn.Module):
    def __init__(self, n_predictions, hidden_dim=128):
        super().__init__()
        self.n_predictions = n_predictions
        self.angle_embeddings = nn.Parameter(torch.randn(n_predictions, hidden_dim))
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim//2), 
            nn.ReLU(),
            nn.Linear(hidden_dim//2, 1)
        )
        nn.init.xavier_uniform_(self.angle_embeddings)
    
    def forward(self, H):
        B, T, D = H.shape
        contexts, attns = [], []
        for a in range(self.n_predictions):
            scores = self.mlp(H + self.angle_embeddings[a]).squeeze(-1)
            w = torch.softmax(scores, dim=-1)
            ctx = (w.unsqueeze(-1) * H).sum(1)
            contexts.append(ctx)
            attns.append(w)
        return torch.stack(contexts, dim=1), torch.stack(attns, dim=1)

class AttentionLSTM(nn.Module):
    def __init__(self, input_features, n_predictions, output_features=1, hidden_dim=128, lstm_layers=2, dropout=0.3):
        super().__init__()
        self.input_features = input_features
        self.n_predictions = n_predictions
        self.output_features = output_features
        self.hidden_dim = hidden_dim
        
        self.lstm = nn.LSTM(
            input_features, 
            hidden_dim, 
            num_layers=lstm_layers, 
            batch_first=True, 
            dropout=dropout if lstm_layers > 1 else 0.0
        )
        self.ln = nn.LayerNorm(hidden_dim)
        self.attention = SimpleMLPAttention(n_predictions, hidden_dim)
        
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.ReLU(),
            nn.Linear(hidden_dim//2, hidden_dim//4),
            nn.ReLU(),
            nn.Linear(hidden_dim//4, output_features)
        )
    
    def forward(self, x):
        o, _ = self.lstm(x)
        o = self.ln(o)
        ctx, attn = self.attention(o)
        out = self.fc(ctx)
        return out, attn

# -------------------------------
# METRICS COMPUTATION
# -------------------------------
def compute_epoch_metrics(y_true, y_pred):
    """Compute comprehensive metrics for a single epoch"""
    y_true_np = y_true.cpu().numpy() if torch.is_tensor(y_true) else y_true
    y_pred_np = y_pred.cpu().numpy() if torch.is_tensor(y_pred) else y_pred
    
    # Flatten for overall metrics
    y_true_flat = y_true_np.flatten()
    y_pred_flat = y_pred_np.flatten()
    
    # Basic metrics
    mse = mean_squared_error(y_true_flat, y_pred_flat)
    mae = mean_absolute_error(y_true_flat, y_pred_flat)
    r2 = r2_score(y_true_flat, y_pred_flat)
    
    # Additional metrics
    rmse = np.sqrt(mse)

    epsilon = 1e-8
    # Use mean of absolute values as denominator to avoid division by near-zero
    denominator = np.abs(y_true_flat) + epsilon
    # Alternative: use max(absolute_value, epsilon) for each element
    # denominator = np.maximum(np.abs(y_true_flat), epsilon)
    
    mape = np.mean(np.abs((y_true_flat - y_pred_flat) / denominator)) * 100
    
    # Max Error
    max_error = np.max(np.abs(y_true_flat - y_pred_flat))
    
    # Explained Variance Score (similar to R² but different calculation)
    from sklearn.metrics import explained_variance_score
    evs = explained_variance_score(y_true_flat, y_pred_flat)
    
    # Mean Bias Error (shows if model systematically over/under predicts)
    mbe = np.mean(y_pred_flat - y_true_flat)
    
    # Median Absolute Error (more robust to outliers than MAE)
    from sklearn.metrics import median_absolute_error
    medae = median_absolute_error(y_true_flat, y_pred_flat)
    
    return {
        'mse': mse,
        'rmse': rmse,
        'mae': mae,
        'r2': r2,
        'mape': mape,
        'max_error': max_error,
        'evs': evs,
        'mbe': mbe,
        'medae': medae
    }

def compute_all_metrics(y_true, y_pred):
    """Compute comprehensive metrics"""
    y_true_np = y_true.cpu().numpy() if torch.is_tensor(y_true) else y_true
    y_pred_np = y_pred.cpu().numpy() if torch.is_tensor(y_pred) else y_pred
    
    mse = mean_squared_error(y_true_np.flatten(), y_pred_np.flatten())
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true_np.flatten(), y_pred_np.flatten())
    r2 = r2_score(y_true_np.flatten(), y_pred_np.flatten())
    
    per_pred_mse = np.mean((y_true_np - y_pred_np)**2, axis=(0, 2))
    per_pred_mae = np.mean(np.abs(y_true_np - y_pred_np), axis=(0, 2))
    
    if y_true_np.ndim == 3 and y_true_np.shape[2] > 1:
        per_feature_mse = np.mean((y_true_np - y_pred_np)**2, axis=(0, 1))
        per_feature_mae = np.mean(np.abs(y_true_np - y_pred_np), axis=(0, 1))
    else:
        per_feature_mse = None
        per_feature_mae = None
    
    max_error = np.max(np.abs(y_true_np - y_pred_np))
    mean_error = np.mean(y_pred_np - y_true_np)
    std_error = np.std(y_pred_np - y_true_np)
    residuals = y_pred_np - y_true_np
    
    metrics = {
        'mse': float(mse),
        'rmse': float(rmse),
        'mae': float(mae),
        'r2': float(r2),
        'max_error': float(max_error),
        'mean_error': float(mean_error),
        'std_error': float(std_error),
        'per_prediction_mse': per_pred_mse.tolist(),
        'per_prediction_mae': per_pred_mae.tolist(),
        'residuals': residuals
    }
    
    if per_feature_mse is not None:
        metrics['per_feature_mse'] = per_feature_mse.tolist()
        metrics['per_feature_mae'] = per_feature_mae.tolist()
    
    return metrics

# -------------------------------
# ORGANIZED IMAGE SAVER CLASS
# -------------------------------
class OrganizedImageSaver:
    def __init__(self, base_dir="images"):
        self.base_dir = Path(base_dir)
        
        # Create four main folders
        self.predictions_dir = self.base_dir / "01_predictions"
        self.loss_dir = self.base_dir / "02_loss"
        self.attention_dir = self.base_dir / "03_attention"
        self.attention_csv_dir = self.base_dir / "03_attention_csv"
        
        self.predictions_dir.mkdir(parents=True, exist_ok=True)
        self.loss_dir.mkdir(parents=True, exist_ok=True)
        self.attention_dir.mkdir(parents=True, exist_ok=True)
        self.attention_csv_dir.mkdir(parents=True, exist_ok=True)
        
        self.epoch_count = 0
    
    def save_epoch_plots(self,sensor_data, pred_data, loss_data, attn_data, epoch, 
                         n_samples, x_axis, y_lim, PREDICTIONS_OUT, TIMESTEPS_IN,
                         train_loss, val_loss, best_val_loss):
        """Save each subplot as a separate image in organized folders"""
        
        # 1. PREDICTIONS PLOT
        fig_pred = plt.figure(figsize=(12, 7))
        ax_pred = fig_pred.add_subplot(111)
        
        true_np, pred_np, idxs = pred_data
        
        # 3. ATTENTION HEATMAP
        attn_mean = attn_data
        attn_path = self.attention_dir / f"attention_epoch_{epoch:04d}.png"
        plot_selected_features_with_attn_heatmap(sensor_data, feature_names, attn_mean, attn_path)
        
        for i, idx in enumerate(idxs):
            ax_pred.plot(x_axis, true_np[idx, :, 0], 'o-', lw=3, ms=9, label=f'True {i}')
            ax_pred.plot(x_axis, pred_np[idx, :, 0], '--s', lw=2.8, ms=7, alpha=0.9, label=f'Pred {i}')
        
        ax_pred.set_ylim(*y_lim)
        ax_pred.set_xlabel(f"Prediction Index (Total: {PREDICTIONS_OUT})", fontsize=12)
        ax_pred.set_ylabel("Target Value (Feature 0)", fontsize=12)
        ax_pred.set_title(f"Predictions - Epoch {epoch} ({n_samples} samples)", fontweight='bold', fontsize=14)
        ax_pred.grid(alpha=0.3)
        ax_pred.legend(loc='upper left', fontsize=9, ncol=2)
        plt.tight_layout()
        
        pred_path = self.predictions_dir / f"predictions_epoch_{epoch:04d}.png"
        fig_pred.savefig(pred_path, dpi=150, bbox_inches='tight')
        plt.close(fig_pred)
        
        # 2. LOSS PLOT
        fig_loss = plt.figure(figsize=(10, 7))
        ax_loss = fig_loss.add_subplot(111)
        
        epochs_list, val_losses, train_losses = loss_data
        
        ax_loss.plot(epochs_list, train_losses, color='#1f77b4', lw=3, alpha=0.7, label='Train MSE')
        ax_loss.plot(epochs_list, val_losses, color='#d62728', lw=3, label='Val MSE')
        ax_loss.plot(epochs_list, [best_val_loss] * len(epochs_list), 
                    color='green', lw=2.5, ls='--', label='Best Val MSE')
        
        ax_loss.set_xlabel("Epoch", fontsize=12)
        ax_loss.set_ylabel("MSE", fontsize=12)
        ax_loss.set_title(f"Training Progress - Epoch {epoch}\nTrain: {train_loss:.6f} | Val: {val_loss:.6f} | Best: {best_val_loss:.6f}", 
                         fontweight='bold', fontsize=14)
        ax_loss.grid(alpha=0.3)
        ax_loss.legend(fontsize=10)
        plt.tight_layout()
        
        loss_path = self.loss_dir / f"loss_epoch_{epoch:04d}.png"
        fig_loss.savefig(loss_path, dpi=150, bbox_inches='tight')
        plt.close(fig_loss)
        
        
        
        # 4. SAVE ATTENTION MATRIX AS CSV
        attn_df = pd.DataFrame(
            attn_mean,
            index=[f"Pred_{i}" for i in range(attn_mean.shape[0])],
            columns=[f"Time_{i}" for i in range(attn_mean.shape[1])]
        )
        
        csv_path = self.attention_csv_dir / f"attention_epoch_{epoch:04d}.csv"
        attn_df.to_csv(csv_path, float_format='%.6f')

        self.epoch_count = epoch
        
        return pred_path, loss_path, attn_path, csv_path

# -------------------------------
# TRAINING WITH ORGANIZED MLFLOW LOGGING
# -------------------------------
def train_model(params):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    with mlflow.start_run(run_name=f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"):
        
        # Create organized image directories
        image_saver = OrganizedImageSaver("images")
        
        mlflow.set_tag("description", EXPERIMENT_DESCRIPTION)
        mlflow.log_param("input_shape", f"({N_EXPERIMENTS}, {TIMESTEPS_IN}, {FEATURES_IN})")
        mlflow.log_param("output_shape", f"({N_EXPERIMENTS}, {PREDICTIONS_OUT}, {FEATURES_OUT})")
        
        mlflow.log_param("num_selected_features", FEATURES_IN)

        mlflow.log_params(params)
        mlflow.log_param("device", str(device))
        mlflow.log_param("input_features", FEATURES_IN)
        mlflow.log_param("output_features", FEATURES_OUT)
        mlflow.log_param("timesteps", TIMESTEPS_IN)
        mlflow.log_param("predictions_out", PREDICTIONS_OUT)
        mlflow.log_param("n_experiments", N_EXPERIMENTS)
        mlflow.log_param("seed", SEED)
        
        # Split data
        X_train, X_val, Y_train, Y_val = train_test_split(
            X_seq, Y_target, test_size=0.2, random_state=42
        )
                
        mlflow.log_param("train_size", len(X_train))
        mlflow.log_param("val_size", len(X_val))
        
        # Compute global y-limits for plotting (using first feature)
        y_all = Y_val[:, :, 0].cpu().numpy()
        global_ymin, global_ymax = y_all.min(), y_all.max()
        margin = (global_ymax - global_ymin) * 0.1
        y_lim = (global_ymin - margin, global_ymax + margin)
        
        # Plotting batch
        val_ds = ProcessDataset(X_val, Y_val)
        plot_loader = DataLoader(val_ds, batch_size=min(64, len(val_ds)), shuffle=False)
        plot_X, plot_Y = next(iter(plot_loader))
        plot_X = plot_X.to(device)
        
        x_axis = np.arange(PREDICTIONS_OUT)
        n_samples = min(5, len(plot_Y))
        
        # Training setup
        train_ds = ProcessDataset(X_train, Y_train)
        train_loader = DataLoader(train_ds, batch_size=params['batch_size'], shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=32)
        
        # Model setup
        model = AttentionLSTM(
            input_features=FEATURES_IN,
            n_predictions=PREDICTIONS_OUT,
            output_features=FEATURES_OUT,
            hidden_dim=params['hidden_dim'],
            lstm_layers=params['lstm_layers'],
            dropout=params['dropout']
        ).to(device)
        
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        mlflow.log_param("total_parameters", total_params)
        mlflow.log_param("trainable_parameters", trainable_params)
        
        optimizer = optim.AdamW(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])
        scheduler = ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
        criterion = nn.MSELoss()
        
        val_losses = []
        train_losses = []
        learning_rates = []
        best_val_loss = float('inf')
        best_state = None
        patience = 0
        epoch_times = []
        
        
        for epoch in range(1, params['max_epochs'] + 1):
            epoch_start = time.time()
            
            # Training
            model.train()
            train_loss = 0.0
            for Xb, Yb in train_loader:
                Xb, Yb = Xb.to(device), Yb.to(device)
                pred, _ = model(Xb)
                loss = criterion(pred, Yb)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()
                train_loss += loss.item()
            train_loss /= len(train_loader)
            train_losses.append(train_loss)
            
            # Validation
            model.eval()
            val_loss = 0.0
            val_preds_epoch = []
            val_targets_epoch = []
            
            with torch.no_grad():
                for Xb, Yb in val_loader:
                    Xb, Yb = Xb.to(device), Yb.to(device)
                    pred, _ = model(Xb)
                    val_loss += criterion(pred, Yb).item()
                    
                    # Collect predictions and targets for metrics
                    val_preds_epoch.append(pred.cpu())
                    val_targets_epoch.append(Yb.cpu())
            
            val_loss /= len(val_loader)
            val_losses.append(val_loss)
            
            # Compute validation metrics
            val_preds_epoch = torch.cat(val_preds_epoch, dim=0)
            val_targets_epoch = torch.cat(val_targets_epoch, dim=0)
            metrics = compute_epoch_metrics(val_targets_epoch, val_preds_epoch)
            
            scheduler.step(val_loss)
            current_lr = optimizer.param_groups[0]['lr']
            learning_rates.append(current_lr)
            
            epoch_time = time.time() - epoch_start
            epoch_times.append(epoch_time)
            
            # Log all metrics to MLflow
            mlflow.log_metrics({
                'train_loss': train_loss,
                'val_loss': val_loss,
                'val_mse': metrics['mse'],
                'val_rmse': metrics['rmse'],
                'val_mae': metrics['mae'],
                'val_r2': metrics['r2'],
                'val_mape': metrics['mape'],
                'val_max_error': metrics['max_error'],
                'val_evs': metrics['evs'],
                'val_mbe': metrics['mbe'],
                'val_medae': metrics['medae'],
                'learning_rate': current_lr,
                'epoch_time': epoch_time
            }, step=epoch)
            
            if val_loss < best_val_loss - 1e-6:
                best_val_loss = val_loss
                best_state = model.state_dict()
                patience = 0
                mlflow.log_metric('best_val_loss', best_val_loss, step=epoch)
            else:
                patience += 1
            
            # Save separate plots every 2 epochs
            if epoch % 2 == 0 or epoch == 1:
                with torch.no_grad():
                    pred, attn = model(plot_X)
                    pred_np = pred.cpu().numpy()
                    true_np = plot_Y.cpu().numpy()
                    attn_mean = attn.mean(0).cpu().numpy()
                
                idxs = random.sample(range(len(true_np)), min(n_samples, len(true_np)))
                
                # Prepare data for plotting
                pred_data = (true_np, pred_np, idxs)
                loss_data = (list(range(1, len(val_losses) + 1)), val_losses, train_losses)
                attn_data = attn_mean
                
                image_saver.save_epoch_plots(
                    X_seq, pred_data, loss_data, attn_data, epoch,
                    n_samples, x_axis, y_lim, PREDICTIONS_OUT, TIMESTEPS_IN,
                    train_loss, val_loss, best_val_loss
                )
            
            # Enhanced print statement with metrics
            print(f"Epoch {epoch:3d} → Train: {train_loss:.6f} | Val: {val_loss:.6f} | "
                  f"MSE: {metrics['mse']:.6f} | MAE: {metrics['mae']:.6f} | R²: {metrics['r2']:.4f} | "
                  f"MAPE: {metrics['mape']:.2f}% | MedAE: {metrics['medae']:.6f} | "
                  f"Best: {best_val_loss:.6f} | LR: {current_lr:.2e}")
            
            if patience >= 10:
                print("\nEarly stopping!")
                mlflow.log_param("stopped_at_epoch", epoch)
                break
        
        # Load best model
        if best_state is not None:
            model.load_state_dict(best_state)
        
        # Final evaluation
        model.eval()
        all_preds, all_targets = [], []
        with torch.no_grad():
            for Xb, Yb in val_loader:
                Xb = Xb.to(device)
                pred, _ = model(Xb)
                all_preds.append(pred.cpu())
                all_targets.append(Yb)
        
        all_preds = torch.cat(all_preds, dim=0)
        all_targets = torch.cat(all_targets, dim=0)
        
        final_metrics = compute_all_metrics(all_targets, all_preds)
        
        metrics_to_log = {
            'final_mse': final_metrics['mse'],
            'final_rmse': final_metrics['rmse'],
            'final_mae': final_metrics['mae'],
            'final_r2': final_metrics['r2'],
            'final_max_error': final_metrics['max_error'],
            'final_mean_error': final_metrics['mean_error'],
            'final_std_error': final_metrics['std_error'],
            'total_epochs': len(val_losses),
            'avg_epoch_time': np.mean(epoch_times)
        }
        
        if 'per_feature_mse' in final_metrics:
            for i, (mse, mae) in enumerate(zip(final_metrics['per_feature_mse'], final_metrics['per_feature_mae'])):
                metrics_to_log[f'final_mse_feature_{i}'] = mse
                metrics_to_log[f'final_mae_feature_{i}'] = mae
        
        mlflow.log_metrics(metrics_to_log)
        
        # Log model
        mlflow.pytorch.log_model(model, "model")
        
        # Log images from the last epoch
        if image_saver.epoch_count > 0:
            mlflow.log_artifact(str(image_saver.predictions_dir / f"predictions_epoch_{image_saver.epoch_count:04d}.png"))
            mlflow.log_artifact(str(image_saver.loss_dir / f"loss_epoch_{image_saver.epoch_count:04d}.png"))
            mlflow.log_artifact(str(image_saver.attention_dir / f"attention_epoch_{image_saver.epoch_count:04d}.png"))
            mlflow.log_artifact(str(image_saver.attention_csv_dir / f"attention_epoch_{image_saver.epoch_count:04d}.csv"))
        
        return {
            'model': model,
            'best_val_loss': best_val_loss,
            'final_metrics': final_metrics
        }


# Define hyperparameters
params = {
    'hidden_dim': 32,      # Reduce model size
    'lstm_layers': 1,      # Simpler architecture  
    'dropout': 0.2,        # Increase regularization
    'lr': 0.0005, # 5e-4,           # Smaller learning rate
    'weight_decay': 1e-3,  # Stronger L2 regularization
    'batch_size': 8,      # Smaller batches
    'max_epochs': 800     # Train longer with good reg
}

# Train model
result = train_model(params)

Main plot timesteps: 200
Attention original shape: (46, 200)
Attention final shape: (46, 200)
Epoch   1 → Train: 0.252429 | Val: 0.225941 | MSE: 0.226375 | MAE: 0.377255 | R²: -1.3103 | MAPE: 61186750.00% | MedAE: 0.328349 | Best: 0.225941 | LR: 5.00e-04
Main plot timesteps: 200
Attention original shape: (46, 200)
Attention final shape: (46, 200)
Epoch   2 → Train: 0.214145 | Val: 0.201754 | MSE: 0.202151 | MAE: 0.350905 | R²: -1.0631 | MAPE: 64715025.00% | MedAE: 0.310380 | Best: 0.201754 | LR: 5.00e-04
Epoch   3 → Train: 0.191983 | Val: 0.178664 | MSE: 0.179000 | MAE: 0.329496 | R²: -0.8268 | MAPE: 70628418.75% | MedAE: 0.274139 | Best: 0.178664 | LR: 5.00e-04
Main plot timesteps: 200
Attention original shape: (46, 200)
Attention final shape: (46, 200)
Epoch   4 → Train: 0.166651 | Val: 0.150119 | MSE: 0.150382 | MAE: 0.303187 | R²: -0.5347 | MAPE: 73126681.25% | MedAE: 0.237304 | Best: 0.150119 | LR: 5.00e-04
Epoch   5 → Train: 0.132762 | Val: 0.106947 | MSE: 0.107089 | MAE: 0.25057

2025/11/26 14:01:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/26 14:01:44 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/11/26 14:01:44 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


# Complete Guide to All Regression Metrics

## 1. **MSE (Mean Squared Error)**

**Formula:** Average of (predicted − actual)²

### What it means

-   Measures the average squared difference between predictions and
    actual values.
-   Squares errors, so large errors are penalized more heavily.
-   Units are squared.

### Example

True: \[10, 20, 30\]\
Pred: \[12, 19, 33\]\
Errors: \[2, −1, 3\]\
Squared: \[4, 1, 9\]\
**MSE = (4 + 1 + 9) / 3 = 4.67**

### When to use

-   When you want to heavily penalize large errors.
-   Standard metric for training neural networks.

### Interpretation

-   Lower is better; 0 = perfect.
-   Harder to interpret because units are squared.

------------------------------------------------------------------------

## 2. **RMSE (Root Mean Squared Error)**

**Formula:** √MSE

### What it means

-   Square root of MSE.
-   Units match the target variable.
-   Still penalizes large errors.

### Example

MSE = 4.67\
**RMSE = √4.67 = 2.16**

### When to use

-   When you want MSE behavior but more interpretable units.

### Interpretation

-   Lower is better.
-   Example: RMSE = 2.16 → predictions are off by about 2.16 units on
    average.

------------------------------------------------------------------------

## 3. **MAE (Mean Absolute Error)**

**Formula:** Average of \|predicted − actual\|

### What it means

-   Average absolute error.
-   All errors treated equally.
-   Most intuitive for humans.

### Example

True: \[10, 20, 30\]\
Pred: \[12, 19, 33\]\
Abs errors: \[2, 1, 3\]\
**MAE = (2 + 1 + 3) / 3 = 2.0**

### When to use

-   You want simple and intuitive interpretation.
-   Large errors should not dominate the metric.

### Interpretation

-   Lower is better.
-   MAE = 2 means predictions are off by 2 units on average.

### MAE vs RMSE

-   MAE ≤ RMSE always.
-   RMSE \>\> MAE → large outliers.
-   RMSE ≈ MAE → consistent errors.

------------------------------------------------------------------------

## 4. **R² (Coefficient of Determination)**

**Formula:** 1 − (sum of squared residuals / total variance)

### What it means

-   Measures % of variance explained by the model.
-   Compares model to baseline (predicting the mean).

### Example

Variance = 100\
Unexplained variance = 20\
**R² = 1 − (20/100) = 0.80**

### When to use

-   Model comparison.
-   Understanding overall goodness-of-fit.

### Interpretation

-   Range: −∞ to 1
-   1.0: perfect\
-   0.9: excellent\
-   0.7--0.8: good\
-   0.5: moderate\
-   0: predicts mean\
-   \<0: worse than mean predictor

------------------------------------------------------------------------

## 5. **MAPE (Mean Absolute Percentage Error)**

**Formula:** Average of \|(predicted − actual) / actual\| × 100%

### What it means

-   Error expressed as percentage.
-   Scale-independent.
-   Intuitive for business contexts.

### Example

True: \[10, 20, 30\]\
Pred: \[12, 19, 33\]\
Percentage errors: \[20%, 5%, 10%\]\
**MAPE = (20 + 5 + 10) / 3 = 11.67%**

### When to use

-   Comparing models across scales.
-   Business reporting.
-   When target values are not near zero.

### Interpretation

-   \<5% excellent\

-   5--10% very good\

-   10--20% good\

-   20--50% reasonable\

-   50% poor

### Warnings

-   Undefined when true value = 0.
-   Sensitive when true values near zero.

------------------------------------------------------------------------

## 6. **Max Error**

**Formula:** Maximum \|predicted − actual\|

### What it means

-   Shows worst-case prediction.

### Example

Errors: \[0.1, 0.3, 5.2, 0.2, 0.4\]\
**Max Error = 5.2**

### When to use

-   Safety‑critical systems.
-   Outlier analysis.

### Interpretation

-   Lower is better.
-   Highly sensitive to outliers.

------------------------------------------------------------------------

## 7. **EVS (Explained Variance Score)**

**Formula:** 1 − (Var(residuals) / Var(true values))

### What it means

-   Measures variance captured by the model.
-   More lenient than R² with bias.

### Interpretation

-   Range: −∞ to 1
-   EVS ≥ R² typically.
-   EVS \>\> R² → model has bias but captures pattern well.

------------------------------------------------------------------------

## 8. **MBE (Mean Bias Error)**

**Formula:** Average(predicted − actual)

### What it means

-   Detects systematic over‑ or under‑prediction.

### Example

Pred: \[12, 21, 32\]\
True: \[10, 20, 30\]\
Errors: \[+2, +1, +2\]\
**MBE = 1.67 (overprediction)**

### Interpretation

-   MBE \> 0: overpredict\
-   MBE \< 0: underpredict\
-   Close to 0: little bias

------------------------------------------------------------------------

## 9. **MedAE (Median Absolute Error)**

**Formula:** Median \|predicted − actual\|

### What it means

-   Outlier‑robust version of MAE.
-   Reports the 50th percentile error.

### Example

Errors: \[0.5, 0.8, 1.0, 1.2, 10.0\]\
**MedAE = 1.0**

### Interpretation

-   Lower is better.
-   Useful when data has outliers.